# Лабораторная работа 2. Эмпирический риск и метод наименьших квадратов

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| Место в курсе | после лекции 1 |
| Опора на лекции | лекция 1: функция потерь (опр. 1.9), эмпирический и истинный риск (опр. 1.10–1.11), утв. 1.12 о несмещённости, принцип ERM (опр. 1.13), связь ERM и ММП (утв. 1.15), матричное дифференцирование (§6), нормальные уравнения МНК (теорема 1.20) |
| Трудоёмкость | 2 ч аудиторно (части 1–4) + 4 ч самостоятельно |

## Цель работы

Превратить формулы лекции 1 в работающий код и проверить их численно: убедиться, что эмпирический риск несмещённо оценивает истинный **только** для алгоритма, выбранного независимо от выборки; проверить формулы матричного дифференцирования конечными разностями; реализовать МНК двумя способами и увидеть, где нормальные уравнения численно проигрывают SVD; убедиться на эксперименте, что оценка максимального правдоподобия при гауссовском шуме совпадает с решением МНК.

## Что нужно сдать

Заполненный ноутбук `lab02_student.ipynb`, в котором:

1. выполнены все задания (ячейки с `# TODO`), код запускается сверху вниз без ошибок;
2. под каждым заданием заполнены ячейки **Вывод** — своими словами, не пересказ кода;
3. в конце — раздел «Итоги работы» с ответами на контрольные вопросы;
4. все графики подписаны (заголовок, оси, легенда).

> **Индивидуальный вариант.** Датасет и набор методов выдаются по вашему ФИО
> (см. ячейку ниже). Отчёт с чужим вариантом не принимается.

Вся работа — это проверка утверждений лекции 1 вычислительным экспериментом.
Формулы даны, доказательства прочитаны; здесь мы отвечаем на вопрос
«а действительно ли так?» и заодно нарабатываем инструменты для работ 3–9.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from scipy import optimize
from labdata import load_personal

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=2)
describe_variant(variant)

---
# Часть 1. Функция потерь и эмпирический риск

По определению 1.9, функция потерь $\mathcal L(a, x, y) \ge 0$ измеряет ошибку
алгоритма $a$ на объекте $x$ с ответом $y$; в регрессии она зависит только от
невязки $r = a(x) - y$. Эмпирический риск (опр. 1.10):

$$
Q(a, X^\ell) \;=\; \frac1\ell \sum_{i=1}^{\ell} \mathcal L(a, x_i, y_i).
$$

Реализуем четыре потери:

$$
\mathcal L_{\mathrm{sq}}(r) = r^2, \qquad
\mathcal L_{\mathrm{abs}}(r) = |r|, \qquad
\mathcal L_{\delta}(r) = \begin{cases} \tfrac12 r^2, & |r| \le \delta,\\[2pt]
\delta\bigl(|r| - \tfrac12\delta\bigr), & |r| > \delta,\end{cases} \qquad
\mathcal L_{q}(r) = \begin{cases} (1-q)\,r, & r \ge 0,\\ -q\,r, & r < 0.\end{cases}
$$

Последние две — потеря Хьюбера (квадратичная вблизи нуля, линейная на хвостах)
и квантильная потеря (асимметричная: перепрогноз $r>0$ и недопрогноз $r<0$
штрафуются с разными весами $1-q$ и $q$).

In [ ]:
# TODO: реализуйте четыре функции потерь от невязки r = a(x) - y
def loss_squared(r):
    raise NotImplementedError


def loss_absolute(r):
    raise NotImplementedError


def loss_huber(r, delta=1.0):
    raise NotImplementedError


def loss_quantile(r, q=0.75):
    raise NotImplementedError


LOSSES = {
    "квадратичная": loss_squared,
    "абсолютная": loss_absolute,
    "Хьюбера": loss_huber,
    "квантильная (q=0.75)": loss_quantile,
}


def empirical_risk(loss, pred, y):
    """Q(a, X^l) -- среднее значение потерь по выборке."""
    raise NotImplementedError


# TODO: постройте графики всех четырёх потерь на одном рисунке
print("ваш вариант:", ", ".join(variant["losses"]))

### Задание 1.2. Оптимальный константный алгоритм

Возьмём простейшую модель алгоритмов — константы: $A = \{a(x) \equiv c \mid c \in \mathbb{R}\}$.
Принцип ERM (опр. 1.13) требует найти

$$
c^* = \arg\min_{c} \; \frac1\ell\sum_{i=1}^{\ell} \mathcal L(c - y_i).
$$

Для каждой из ваших двух потерь найдите $c^*$ численно (перебором по сетке или
`scipy.optimize.minimize_scalar`) и сравните с выборочными характеристиками:
средним, медианой и квантилями вектора $y$. Постройте график $Q(c)$.

**Это задание объясняет, почему в работе 1 константный ориентир для MSE брался
средним, а не медианой.**

In [ ]:
y_sample = rng.gamma(shape=2.0, scale=3.0, size=400)     # заведомо несимметричное

# TODO: для каждой потери из variant["losses"]:
#   1) постройте график Q(c) на сетке значений c;
#   2) найдите c* численно (minimize_scalar с bounds или argmin по сетке);
#   3) сравните c* со средним, медианой и квантилью 0.75 вектора y_sample.

> **Вывод.** Какой выборочной характеристике равен $c^*$ для каждой из ваших потерь? Почему для квадратичной потери это именно среднее? (Продифференцируйте $Q(c)$ по $c$ и приравняйте нулю.)
>
> *(ваш ответ здесь)*

---
# Часть 2. Эмпирический риск против истинного: проверяем утверждение 1.12

Утверждение 1.12 гласит: если $X^\ell$ — простая выборка, то
$\mathbb{E}\bigl[Q(a, X^\ell)\bigr] = R(a)$.

Ключевое слово — **алгоритм $a$ фиксирован до того, как увидена выборка**.
В доказательстве использовано, что $\mathcal L(a, x_i, y_i)$ одинаково
распределены; если $a$ сам зависит от $X^\ell$, это рассуждение рушится.

Проверим оба случая на модели, где всё считается точно:

$$
x \sim U[0, 1], \qquad y = \theta_0 + \theta_1 x + \varepsilon,
\qquad \varepsilon \sim \mathcal N(0, \sigma^2).
$$

Для истинного алгоритма $a^*(x) = \theta_0 + \theta_1 x$ истинный риск при
квадратичной потере равен в точности
$R(a^*) = \mathbb{E}(a^*(x) - y)^2 = \mathbb{E}\varepsilon^2 = \sigma^2$.

In [ ]:
THETA_TRUE = np.array([2.0, 3.0])          # свободный член и наклон
SIGMA = variant["noise_sigma"]             # из вашего варианта
ELL = 30                                   # размер обучающей выборки
N_REPEAT = 3000                            # число повторений эксперимента


def sample(n, generator):
    """Простая выборка (опр. 1.7) из описанного распределения p(x, y)."""
    x = generator.uniform(0, 1, n)
    X = np.column_stack([np.ones(n), x])
    y = X @ THETA_TRUE + generator.normal(0, SIGMA, n)
    return X, y


print(f"sigma = {SIGMA}, истинный риск R(a*) = sigma^2 = {SIGMA**2:.4f}")

### Задание 2.1. Несмещённость для фиксированного алгоритма

Повторите $N$ раз: сгенерируйте выборку размера $\ell$ и вычислите
$Q(a^*, X^\ell)$ для **истинного** алгоритма. Постройте гистограмму полученных
значений, отметьте на ней $\sigma^2$ и среднее по повторениям.

In [ ]:
gen = np.random.default_rng(RANDOM_STATE)

# TODO: N_REPEAT раз сгенерируйте выборку размера ELL и посчитайте Q(a*, X^l)
#       для истинного алгоритма a*(x) = THETA_TRUE[0] + THETA_TRUE[1] * x.
# TODO: гистограмма Q, вертикальные линии в sigma^2 и в среднем значении Q.
# TODO: напечатайте относительное смещение и стандартное отклонение Q.

### Задание 2.2. Что происходит, если алгоритм выбран по той же выборке

Теперь на каждой выборке решаем задачу ERM: подбираем $\hat\theta$ методом
наименьших квадратов и считаем **три** величины:

* $Q(\hat a, X^\ell)$ — эмпирический риск на **той же** выборке (обучающая ошибка);
* $\hat R(\hat a)$ — оценка истинного риска на большой независимой выборке;
* $\sigma^2$ — неустранимый уровень шума.

Для линейной регрессии с $p$ параметрами теория даёт точные ответы:

$$
\mathbb{E}\,Q(\hat a, X^\ell) = \sigma^2\Bigl(1 - \frac{p}{\ell}\Bigr),
\qquad
\mathbb{E}\,R(\hat a) \approx \sigma^2\Bigl(1 + \frac{p}{\ell}\Bigr).
$$

Проверьте обе формулы численно.

In [ ]:
gen = np.random.default_rng(RANDOM_STATE + 1)
X_big, y_big = sample(200_000, gen)          # «генеральная совокупность» для оценки R

# TODO: N_REPEAT раз:
#   1) сгенерируйте выборку размера ELL;
#   2) решите на ней задачу ERM (np.linalg.lstsq) -> theta_hat;
#   3) посчитайте Q(a_hat) на этой же выборке и R(a_hat) на большой выборке.
# TODO: сравните средние с теоретическими sigma^2 (1 -+ p/l), p = 2.
# TODO: постройте две гистограммы на одном рисунке.

> **Вывод.** Почему для фиксированного алгоритма смещения нет, а для обученного — есть? В каком месте доказательства утверждения 1.12 ломается рассуждение? Как зазор $R - Q$ зависит от $\ell$ и от числа параметров $p$?
>
> *(ваш ответ здесь)*

---
# Часть 3. Матричное дифференцирование: проверка формул

В §6 лекции 1 выведены два тождества:

$$
\nabla_w (a^{\mathsf T} w) = a,
\qquad
\nabla_w (w^{\mathsf T} A w) = (A + A^{\mathsf T}) w,
$$

и из них — градиент квадрата нормы невязки:

$$
\nabla_\theta \|X\theta - y\|^2 = 2 X^{\mathsf T}(X\theta - y).
$$

Проверим их **численно**: сравним аналитический градиент с центральной конечной
разностью

$$
\frac{\partial f}{\partial w_k} \approx
\frac{f(w + h e_k) - f(w - h e_k)}{2h}.
$$

Приём стоит запомнить: при реализации любого градиента «с нуля» (работы 3, 4)
эта проверка за пять строк ловит ошибку в выкладке.

In [ ]:
def numeric_grad(f, w, h=1e-6):
    """Градиент центральными разностями."""
    # TODO: реализуйте покоординатную центральную разность
    raise NotImplementedError


n = 6
w0 = rng.normal(size=n)
a = rng.normal(size=n)
A = rng.normal(size=(n, n))                       # намеренно несимметричная

# TODO: проверьте три формулы, сравнив аналитический градиент с численным:
#   f(w) = a^T w             -> grad = a
#   f(w) = w^T A w           -> grad = (A + A^T) w
#   f(w) = ||X w - y||^2     -> grad = 2 X^T (X w - y)

> **Вывод.** Почему в формуле $\nabla_w(w^{\mathsf T}Aw) = (A + A^{\mathsf T})w$ важна несимметричность $A$? Что будет с точностью численного градиента при $h = 10^{-12}$?
>
> *(ваш ответ здесь)*

---
# Часть 4. МНК: нормальные уравнения и SVD

Теорема 1.20: если $X^{\mathsf T}X$ невырождена, то

$$
\theta^* = (X^{\mathsf T}X)^{-1} X^{\mathsf T} y.
$$

Формулу с обратной матрицей **никогда не реализуют буквально**. Сравним пять
способов решить одну и ту же задачу:

1. `np.linalg.inv(X.T @ X) @ X.T @ y` — прямое обращение (так делать не надо);
2. `np.linalg.solve(X.T @ X, X.T @ y)` — разложение Холецкого нормальных уравнений;
3. QR-разложение $X = QR$, затем $R\theta = Q^{\mathsf T}y$;
4. `np.linalg.lstsq` — SVD-решение (то же, что псевдообратная матрица);
5. SVD вручную: $X = U\Sigma V^{\mathsf T} \Rightarrow \theta^* = V\Sigma^{+}U^{\mathsf T}y$.

Ваш вариант требует реализовать два из них (см. `variant["solver_pair"]`),
но полезно написать все.

In [ ]:
from scipy import linalg


def fit_inv(X, y):
    """(X^T X)^{-1} X^T y -- буквально по формуле теоремы 1.20."""
    raise NotImplementedError


def fit_solve(X, y):
    """Решение нормальных уравнений X^T X theta = X^T y без обращения."""
    raise NotImplementedError


def fit_qr(X, y):
    """X = QR, R theta = Q^T y."""
    raise NotImplementedError


def fit_lstsq(X, y):
    """SVD-решение средствами numpy."""
    raise NotImplementedError


def fit_svd(X, y, tol=1e-12):
    """SVD вручную: theta = V Sigma^+ U^T y."""
    raise NotImplementedError


SOLVERS = {
    "нормальные уравнения через обращение (np.linalg.inv)": fit_inv,
    "нормальные уравнения (np.linalg.solve)": fit_solve,
    "QR-разложение (scipy.linalg.qr)": fit_qr,
    "SVD (np.linalg.lstsq)": fit_lstsq,
    "SVD (np.linalg.svd вручную)": fit_svd,
}
print("ваш вариант:", " | ".join(variant["solver_pair"]))

### Задание 4.1. Сверка с примером из лекции

В примере 1.21 лекции 1 МНК применён к выборке из восьми кошек:
$f_1$ — число банок корма в день, $f_2$ — возраст, $y$ — вес.
Лекция даёт $\theta^* \approx (2.201,\ 0.893,\ 0.114)$.

Воспроизведите этот результат всеми пятью способами.

In [ ]:
# Данные примера 1.5 лекции 1
f1 = np.array([1, 2, 2, 3, 2, 3, 1, 4], dtype=float)
f2 = np.array([1, 2, 3, 5, 7, 10, 2, 8], dtype=float)
y_cats = np.array([3.0, 4.2, 4.5, 5.5, 4.8, 6.0, 3.4, 6.6])

# TODO: соберите матрицу X_cats со столбцом единиц, примените все пять решателей,
#       сравните с theta* = (2.201, 0.893, 0.114) из примера 1.21 лекции.
# TODO: выведите cond(X^T X) и cond(X).

### Задание 4.2. Когда нормальные уравнения ломаются

Число обусловленности связано соотношением
$\mathrm{cond}(X^{\mathsf T}X) = \mathrm{cond}(X)^2$: переход к нормальным
уравнениям **возводит обусловленность в квадрат**, то есть теряет вдвое больше
верных знаков.

Проверьте это на матрице Вандермонда $X_{ij} = t_i^{\,j-1}$, $j = 1,\dots,p$ —
она печально известна своей обусловленностью. Постройте график ошибки
$\|\hat\theta - \theta_{\text{истина}}\|$ для всех пяти методов
в зависимости от $p$ (число столбцов) в логарифмическом масштабе.

In [ ]:
t = np.linspace(0, 1, 120)
degrees = range(2, 15)

# TODO: для каждого p:
#   1) X_v = np.vander(t, p, increasing=True), theta_true случайный, y_v = X_v @ theta_true;
#   2) решите задачу всеми пятью методами и запомните ||theta_hat - theta_true||;
#   3) запомните cond(X_v).
# TODO: два графика в логарифмическом масштабе: ошибка методов и cond(X), cond(X)^2.

> **Вывод.** Начиная с какого $p$ метод с явным обращением даёт бессмысленный ответ? Как это связано с машинной точностью $\varepsilon \approx 2\cdot10^{-16}$ и с равенством $\mathrm{cond}(X^{\mathsf T}X) = \mathrm{cond}(X)^2$?
>
> *(ваш ответ здесь)*

---
# Часть 5. Максимум правдоподобия и выбор функции потерь

Утверждение 1.15 связывает ERM и ММП: если положить
$\mathcal L(a_\theta, x, y) = -\ln\varphi(x, y, \theta)$, то принципы совпадают.
Для линейной модели с гауссовским шумом (пример 1.22)

$$
-\ln\varphi = C + \frac{1}{2\sigma^2}\bigl(g(x,\theta) - y\bigr)^2
\;\Longrightarrow\;
\theta_{\mathrm{ML}} = \arg\min_\theta \|X\theta - y\|^2 = \theta^*_{\mathrm{МНК}} .
$$

А если шум лапласовский, $p(\varepsilon) \propto e^{-|\varepsilon| / b}$, то
$-\ln\varphi = C + |g(x,\theta) - y| / b$, и ММП приводит к **абсолютной**
потере, то есть к медианной регрессии.

Проверим оба утверждения численно и посмотрим, что происходит при выбросах.

In [ ]:
def neg_loglik_gauss(theta, X, y, sigma=1.0):
    """-ln L(theta) для модели y = X theta + N(0, sigma^2)."""
    raise NotImplementedError


def neg_loglik_laplace(theta, X, y, b=1.0):
    """-ln L(theta) для модели y = X theta + Laplace(0, b)."""
    raise NotImplementedError


n = 120
x = np.sort(rng.uniform(0, 1, n))
X_lin = np.column_stack([np.ones(n), x])
y_clean = X_lin @ THETA_TRUE + rng.normal(0, 0.3, n)
y_dirty = y_clean.copy()
outlier_idx = rng.choice(n, size=6, replace=False)
y_dirty[outlier_idx] += rng.choice([-1, 1], 6) * rng.uniform(6, 12, 6)

# TODO: 1) для y_clean и для y_dirty найдите theta тремя способами:
#          МНК (fit_lstsq), минимизацией neg_loglik_gauss, минимизацией neg_loglik_laplace
#          (для лапласовской используйте method="Nelder-Mead" -- функция негладкая);
#       2) сведите результаты в таблицу и сравните с THETA_TRUE;
#       3) на диаграмме рассеяния покажите обе прямые и истинную зависимость.

> **Вывод.** Совпали ли ММП при гауссовском шуме и МНК? Какая из двух прямых устояла против выбросов и почему? Как это связано с частью 1 работы?
>
> *(ваш ответ здесь)*

---
# Часть 6. Полиномиальная регрессия: первое переобучение

Как отмечено в примере 1.4 лекции 1, признаками линейной модели могут быть любые
функции исходных данных: при $f_j(x) = x^{\,j-1}$ та же формула
$g(x,\theta) = \sum_j \theta_j f_j(x)$ задаёт полином степени $n - 1$.
Модель остаётся **линейной по параметрам**, поэтому весь аппарат МНК применим
без изменений.

Постройте полиномы степеней из вашего варианта, обучив их своей реализацией МНК,
и сравните эмпирический риск на обучающей и на контрольной выборке.

In [ ]:
def poly_design(x, degree):
    """Матрица объектов-признаков для полинома: столбцы 1, x, x^2, ..., x^degree."""
    raise NotImplementedError


n_train, n_test = 30, 500
x_tr = np.sort(rng.uniform(-1, 1, n_train))
x_te = np.sort(rng.uniform(-1, 1, n_test))
true_f = lambda t: np.sin(3 * t) + 0.5 * t
y_tr = true_f(x_tr) + rng.normal(0, 0.25, n_train)
y_te = true_f(x_te) + rng.normal(0, 0.25, n_test)

# TODO: для каждой степени из variant["poly_degrees"]:
#   обучите полином своей реализацией МНК, посчитайте Q на обучающей и контрольной
#   выборках, запомните max|theta_j|.
# TODO: два графика: (а) данные и подогнанные полиномы; (б) Q(степень) для обеих
#       выборок в логарифмическом масштабе по оси Q.

> **Вывод.** Как ведут себя обучающая и контрольная ошибки с ростом степени? Что происходит с величиной коэффициентов $\max_j |\theta_j|$? Какая степень оптимальна и можно ли было выбрать её по обучающей ошибке?
>
> *(ваш ответ здесь)*

---
# Часть 7. Своя выборка

Примените свою реализацию МНК к индивидуальной выборке, подготовленной в работе 1.
Функция `load_personal` повторяет эталонную предобработку из первой работы
(приведение типов, чистка категорий, удаление дубликатов, вырожденных признаков
и `id`, разбиение и кодирование), чтобы результаты не зависели от аккуратности
вашего кода в работе 1.

Для задачи классификации МНК применяется к индикатору класса, а ответ берётся
по порогу $0.5$; это законная, но не лучшая конструкция — корректный подход
(логистическая регрессия) будет на лекции 3.

In [ ]:
from sklearn.linear_model import LinearRegression

data = load_personal(variant)
Xtr, Xte = data["X_train"], data["X_test"]
ytr, yte = data["y_train"], data["y_test"]
print(f"{data['domain']}: {data['task']}, обучающая {Xtr.shape}, контрольная {Xte.shape}")

# TODO: 1) добавьте столбец единиц и обучите свою реализацию МНК;
#       2) сверьте со sklearn.linear_model.LinearRegression ОТДЕЛЬНО предсказания
#          и коэффициенты; заодно выведите np.linalg.matrix_rank(Atr) и число
#          столбцов Atr -- расхождение объяснится само;
#       3) сравните Q на обучающей и контрольной выборке с константным ориентиром,
#          посчитайте R^2 на контроле;
#       4) если задача классификационная -- посчитайте долю правильных ответов
#          при пороге 0.5 и долю предсказаний, вышедших за отрезок [0, 1].

> **Вывод.** Предсказания вашей реализации и `sklearn` совпали, а коэффициенты — нет. Сравните ранг матрицы с числом её столбцов и объясните, почему так вышло и почему теорема 1.20 здесь неприменима.
>
> *(ваш ответ здесь)*

> **Вывод.** Насколько МНК лучше константы на вашей выборке? Если у вас задача классификации — какая доля предсказаний вышла за отрезок $[0,1]$ и почему это плохо?
>
> *(ваш ответ здесь)*

## Итоги работы

Ответьте письменно на контрольные вопросы (по 2–4 предложения):

1. Утверждение 1.12 говорит, что $\mathbb{E}[Q] = R$. Почему это не противоречит переобучению? Сформулируйте условие, при котором равенство верно.
2. Вы получили $Q(\hat a, X^\ell) = 0$. Что это говорит о качестве алгоритма на новых объектах? Приведите пример алгоритма с нулевым эмпирическим риском и максимально плохим истинным.
3. Почему $\theta^* = (X^{\mathsf T}X)^{-1}X^{\mathsf T}y$ — правильная формула, но неправильная реализация?
4. Как связаны выбор функции потерь и предположение о распределении шума? Какой потере соответствует шум Лапласа и какую характеристику условного распределения $y$ она восстанавливает?
5. Модель полиномов степени $\le d$ вложена в модель степени $\le d+1$. Докажите, что минимум эмпирического риска по большей модели не больше, чем по меньшей. Почему из этого **не** следует, что большая модель лучше?

### Домашнее задание

1. **Взвешенный МНК.** Пусть объекты имеют веса $w_i > 0$ и минимизируется $Q(\theta) = \sum_i w_i (\langle \theta, x_i\rangle - y_i)^2$. Выведите нормальные уравнения (через матричное дифференцирование, как в §6 лекции 1) и покажите, что решение равно $\theta^* = (X^{\mathsf T}WX)^{-1}X^{\mathsf T}Wy$, где $W = \mathrm{diag}(w_i)$. Реализуйте и проверьте: (а) совпадение со `sklearn` при `sample_weight`; (б) что при $w_i = 1/\sigma_i^2$ и гетероскедастичном шуме $\varepsilon_i \sim \mathcal N(0, \sigma_i^2)$ взвешенный МНК точнее обычного (сравните по 500 повторениям).

2. **Свой `numeric_grad` для матричного аргумента.** Обобщите проверку градиента на функции матричного аргумента $f(W)$, $W \in \mathbb{R}^{m\times n}$, и проверьте формулы $\nabla_W \mathrm{tr}(A^{\mathsf T}W) = A$ и $\nabla_W \|XW - Y\|_F^2 = 2X^{\mathsf T}(XW - Y)$. Вторая формула — это многомерная линейная регрессия (несколько целевых переменных сразу); проверьте, что она сводится к независимому решению задач по столбцам $Y$.